# ST-OMR Meter V5-1 — 30 TRAIN BBox Pilot

**Güvenlik sınırı:** Bu notebook yalnız `METER_V2_1500_PACKAGE_AB_CLEAN` veri setini kabul eder. Authoritative dataset yolu `MyDrive/TEST` altında sabitlenmiştir; aynı isimli başka klasörler seçilmez. Önce 1500 yapısal/manifest gate'i arka planda çalışır ve ayrı bir status dosyasına heartbeat yazar, ardından `final_holdout` kilitlenir. Annotation UI yalnız TRAIN içinden deterministik 10×2/4 + 10×3/4 + 10×4/4 örneği açar. Original `image.png` dosyaları değiştirilmez. Training, tuning, checkpoint/model loading ve inference bu notebookta yoktur.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess, sys, shutil
EXPECTED_CODE_SHA = '0146ecae4f3abb06175872bebc6b7f15644b4773'
BRANCH = 'fix/meter-v5-1-clean-bbox-pilot'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_DIR = pathlib.Path('/content/st-omr-training-v5-1')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
actual = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert actual == EXPECTED_CODE_SHA, (actual, EXPECTED_CODE_SHA)
subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('CODE_PIN=PASS', actual)


## Authoritative Drive binding + background precheck

Bu sürüm Drive'ın tamamında isim araması yapmaz. Yalnız doğrulanmış `TEST/METER_V2_1500_PACKAGE_AB_CLEAN` yolunu kullanır. Üst seviyede aynı isimli başka bir klasör varsa yalnız uyarı verir ve onu asla seçmez. Uzun precheck ayrı bir Python thread'inde çalışır; ilerleme `/MyDrive/TEST/ST_OMR_RUNS/METER_V5_1_PRECHECK/V5_1_PRECHECK_STATUS.json` dosyasına atomik olarak yazılır. Sonraki hücre canlı izleme ekranıdır.


In [ ]:
import json, threading, time, traceback
from datetime import datetime, timezone
from pathlib import Path
from st_omr_training.meter_v5_1_bbox_pilot import (
    verify_dataset_structure, ensure_final_holdout_lock, MeterV5_1PilotError,
)

EXPECTED_DATASET_FOLDER_ID = '1OY6ZInOGh0Xtqb4Lkw78zx2rZTfpUasG'
EXPECTED_TEST_FOLDER_ID = '13PKxIyjRgZRhqF1C9n9XRMQCYFn-aSdX'
DATA_ROOT = Path('/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN')
DUPLICATE_TOP_LEVEL = Path('/content/drive/MyDrive/METER_V2_1500_PACKAGE_AB_CLEAN')
MONITOR_ROOT = Path('/content/drive/MyDrive/TEST/ST_OMR_RUNS/METER_V5_1_PRECHECK')
STATUS_PATH = MONITOR_ROOT / 'V5_1_PRECHECK_STATUS.json'

if not DATA_ROOT.is_dir():
    raise MeterV5_1PilotError(f'authoritative dataset path missing: {DATA_ROOT}')
if DATA_ROOT.name != 'METER_V2_1500_PACKAGE_AB_CLEAN' or DATA_ROOT.parent.name != 'TEST':
    raise MeterV5_1PilotError(f'authoritative dataset path binding failed: {DATA_ROOT}')
if DUPLICATE_TOP_LEVEL.exists():
    print('DUPLICATE_DATASET_WARNING=IGNORED', DUPLICATE_TOP_LEVEL)
print('AUTHORITATIVE_DATASET_PATH=PASS', DATA_ROOT)
print('EXPECTED_DATASET_FOLDER_ID=', EXPECTED_DATASET_FOLDER_ID)
print('EXPECTED_TEST_FOLDER_ID=', EXPECTED_TEST_FOLDER_ID)

MONITOR_ROOT.mkdir(parents=True, exist_ok=True)
PRECHECK_STARTED_MONO = time.monotonic()

def _utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')

def _write_status(state, phase, **extra):
    payload = {
        'schema': 'st-omr-meter-v5-1-precheck-status-v1',
        'state': state,
        'phase': phase,
        'updated_utc': _utc_now(),
        'elapsed_seconds': int(time.monotonic() - PRECHECK_STARTED_MONO),
        'dataset_path': str(DATA_ROOT),
        'expected_dataset_folder_id': EXPECTED_DATASET_FOLDER_ID,
        'expected_test_folder_id': EXPECTED_TEST_FOLDER_ID,
        'final_holdout_locked': True,
        'annotation_scope': 'train_pilot_30_only',
        'model_opened': False,
        'training': False,
        'tuning': False,
        'inference_count': 0,
        **extra,
    }
    tmp = STATUS_PATH.with_name(STATUS_PATH.name + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    os.replace(str(tmp), str(STATUS_PATH))
    return payload

def _background_precheck():
    try:
        _write_status('RUNNING', 'VERIFY_DATASET_STRUCTURE')
        gate = verify_dataset_structure(DATA_ROOT)
        _write_status('RUNNING', 'WRITE_FINAL_HOLDOUT_LOCK', dataset_fingerprint_sha256=gate['dataset_fingerprint_sha256'])
        lock_path = ensure_final_holdout_lock(DATA_ROOT, gate)
        globals()['GATE'] = gate
        globals()['LOCK_PATH'] = lock_path
        _write_status(
            'PASS', 'READY_FOR_ANNOTATION',
            dataset_fingerprint_sha256=gate['dataset_fingerprint_sha256'],
            total=gate['total'],
            unique_family_id=gate['unique_family_id'],
            unique_sample_id=gate['unique_sample_id'],
            unique_source_image=gate['unique_source_image'],
            package_ab_only=gate['package_ab_only'],
            cross_split_family_leakage=gate['cross_split_family_leakage'],
            cross_meter_family_overlap=gate['cross_meter_family_overlap'],
        )
    except Exception as exc:
        globals()['PRECHECK_ERROR'] = exc
        _write_status('FAIL', 'BLOCKED', error_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc()[-4000:])

PRECHECK_THREAD = threading.Thread(target=_background_precheck, name='meter-v5-1-precheck', daemon=False)
PRECHECK_THREAD.start()
print('BACKGROUND_PRECHECK=STARTED')
print('STATUS_PATH=', STATUS_PATH)
print('NEXT=Run PRECHECK_MONITOR cell')


## PRECHECK_MONITOR — canlı izleme ekranı

Bu hücre status dosyasını yaklaşık 2 saniyede bir yeniler. Tarayıcı sekmesi kapansa bile Colab runtime yaşamaya devam ettiği sürece arka plan precheck thread'i çalışmayı sürdürür. `PASS` görülmeden annotation UI hücresini çalıştırma.


In [ ]:
from IPython.display import clear_output
print('PRECHECK_MONITOR=STARTED')
last = None
while True:
    try:
        last = json.loads(STATUS_PATH.read_text(encoding='utf-8'))
    except Exception as exc:
        last = {'state': 'WAITING', 'phase': 'STATUS_FILE', 'elapsed_seconds': 0, 'error': str(exc)}
    clear_output(wait=True)
    print('=== ST-OMR METER V5-1 PRECHECK MONITOR ===')
    print('STATE :', last.get('state'))
    print('PHASE :', last.get('phase'))
    print('ELAPSED:', last.get('elapsed_seconds'), 'sec')
    print('DATASET:', last.get('dataset_path', DATA_ROOT))
    print('FINAL_HOLDOUT: LOCKED')
    print('MODEL: CLOSED | TRAINING: CLOSED | INFERENCE: 0')
    if last.get('error'):
        print('ERROR:', last.get('error'))
    if last.get('state') in ('PASS', 'FAIL'):
        break
    time.sleep(2)

if last.get('state') != 'PASS':
    raise MeterV5_1PilotError('background precheck failed; inspect monitor status before continuing')
if 'GATE' not in globals() or 'LOCK_PATH' not in globals():
    raise MeterV5_1PilotError('current runtime does not hold verified precheck objects; rerun background precheck')
print('DATASET_GATE=PASS')
print('data_root=', DATA_ROOT)
print('total=', GATE['total'])
print('unique_family_id=', GATE['unique_family_id'])
print('unique_sample_id=', GATE['unique_sample_id'])
print('unique_source_image=', GATE['unique_source_image'])
print('package_ab_only=', GATE['package_ab_only'])
print('cross_split_family_leakage=', GATE['cross_split_family_leakage'])
print('cross_meter_family_overlap=', GATE['cross_meter_family_overlap'])
for split in ('train','val','final_holdout'):
    print(split.upper(), '2/4=', GATE['directory_counts'][f'{split}/2_4'],
          '3/4=', GATE['directory_counts'][f'{split}/3_4'],
          '4/4=', GATE['directory_counts'][f'{split}/4_4'])
print('final_holdout_locked=', GATE['final_holdout_locked'])
print('final_holdout_lock=', LOCK_PATH)
print('ANNOTATION_SCOPE=train_pilot_30_only')
print('MODEL_OPENED=False; TRAINING=False; TUNING=False; INFERENCE_COUNT=0')


## 30 TRAIN interaktif BBox pilotu

Monitor `STATE: PASS` ve `DATASET_GATE=PASS` gösterdikten sonra bu hücreyi çalıştır. Mouse ile **tek kutu** çiz: üst+alt meter rakamlarının tamamını kapsa; clef/key signature/ilk nota mümkün olduğunca dışarıda kalsın. `KAYDET VE SONRAKİ` PASS olarak checkpoint eder. Emin olmadığın örnekte `REVIEW / ATLA` kullan. Her kayıt anında Drive'daki `annotations/bbox_pilot_30.csv` dosyasına atomik checkpoint edilir ve runtime kapanırsa sample_id üzerinden devam eder.


In [ ]:
if 'GATE' not in globals() or 'LOCK_PATH' not in globals():
    raise MeterV5_1PilotError('PRECHECK PASS required in this runtime before annotation UI')
from st_omr_training.meter_v5_1_bbox_pilot_colab import launch_colab_pilot
SESSION = launch_colab_pilot(data_root=str(DATA_ROOT))
print('PILOT_UI=READY')
print('resume_index=', SESSION.resume_index())
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## Pilot audit — yalnız 30 örnek tamamlandıktan sonra

UI'da 30/30 işlendikten sonra bu hücreyi çalıştır. Audit insan bbox'larını değiştirmez; yalnız mekanik kontrolleri ve boyut istatistiklerini raporlar.


In [ ]:
import json
from st_omr_training.meter_v5_1_bbox_pilot import write_pilot_audit
AUDIT_PATH = write_pilot_audit(DATA_ROOT)
AUDIT = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
print(json.dumps(AUDIT, indent=2, sort_keys=True))
print('AUDIT_PATH=', AUDIT_PATH)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')
